# F04 — Detector scores for the ANES survey study, stage S2 (Study 3)

Scores every human and synthetic respondent document with the two neural detectors of the
first submission: the RoBERTa OpenAI detector (zero-shot, as in Study 2) and Fast-DetectGPT
with GPT-Neo-2.7B (as in Study 1). Thresholds, the TF-IDF detector and the evaluation pools
are built afterwards in the repository from these scores, following the frozen protocol.

**Runtime: GPU (T4), about 40-60 minutes** for about 9,700 documents. Run it once, after
F03 has been run for all three models. Scores are cached on Drive every 250 documents;
after a disconnection run all cells again.

**Input (Drive folder `d2vi_F03`):** the two ANES files and the three
`anes_synth_<model>.parquet` files written by F03.
**Output:** `anes_detector_scores.parquet` + manifest (ids and scores only, no text).


In [ ]:
!pip -q install -U transformers accelerate openpyxl
!pip -q install --force-reinstall --no-deps "d2vi @ git+https://github.com/DRuanli/dvai"
import torch, d2vi
print("d2vi", d2vi.__version__, "| GPU:", torch.cuda.is_available())
assert d2vi.__version__ >= "2.0.0.dev3", "old d2vi still loaded: restart the runtime and run all"


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, glob, json
import numpy as np, pandas as pd
WORK = "/content/drive/MyDrive/d2vi_F03"
print(sorted(os.listdir(WORK)))


In [ ]:
from d2vi.data.anes import ANES2024 as ANESFiles
csv = (sorted(glob.glob(f"{WORK}/anes_timeseries_2024_csv*.zip")) or sorted(glob.glob(f"{WORK}/anes_timeseries_2024_csv*.csv")))[-1]
xlsx = sorted(glob.glob(f"{WORK}/anes_timeseries_2024_redactedopenends*.xlsx"))[-1]
human = ANESFiles(csv, xlsx).human_pool()
assert len(human) == 4926
synth_files = sorted(glob.glob(f"{WORK}/anes_synth_*.parquet"))
print("synthetic files:", [os.path.basename(f) for f in synth_files])
assert len(synth_files) >= 3, "run F03 for the three models first"
synth = pd.concat([pd.read_parquet(f) for f in synth_files], ignore_index=True)
synth = synth[synth.parse_ok & synth.text.fillna("").str.len().gt(0)]
docs = pd.concat([
    pd.DataFrame({"doc_id": human.doc_id, "a": 0.0, "source": "human", "text": human.text}),
    pd.DataFrame({"doc_id": synth.resp_id, "a": 1.0, "source": synth.generator, "text": synth.text}),
], ignore_index=True)
assert docs.doc_id.is_unique
print(docs.groupby("source").size())


In [ ]:
from d2vi.features.neural import RobertaOpenAIDetector, FastDetectGPT
CACHE = f"{WORK}/f04_scores_cache.csv"
done = pd.read_csv(CACHE) if os.path.exists(CACHE) else pd.DataFrame(columns=["doc_id", "score_roberta", "score_fdgpt"])
todo = docs[~docs.doc_id.isin(done.doc_id)]
print("cached", len(done), "| to score", len(todo))
if len(todo):
    rob = RobertaOpenAIDetector(batch_size=64)
    fd = FastDetectGPT()
    for s in range(0, len(todo), 250):
        part = todo.iloc[s:s + 250]
        r = rob.predict(part.text.tolist())
        f = [fd.score_one(t) for t in part.text]
        done = pd.concat([done, pd.DataFrame({"doc_id": part.doc_id, "score_roberta": r, "score_fdgpt": f})], ignore_index=True)
        done.to_csv(CACHE, index=False)
        print(len(done), "scored")


In [ ]:
from d2vi.features.pipeline import package_versions
scores = docs[["doc_id", "a", "source"]].merge(done, on="doc_id", how="left")
assert scores.score_roberta.notna().all()
print(scores.groupby("source")[["score_roberta", "score_fdgpt"]].mean().round(3))
print("Fast-DetectGPT unscorable (< 20 tokens):", scores.score_fdgpt.isna().groupby(scores.source).mean().round(3).to_dict())
out = f"{WORK}/anes_detector_scores.parquet"
scores.to_parquet(out, index=False)
manifest = {"rows": len(scores), "by_source": scores.source.value_counts().to_dict(),
            "detectors": {"score_roberta": RobertaOpenAIDetector.model_id,
                          "score_fdgpt": "Fast-DetectGPT analytic, EleutherAI/gpt-neo-2.7B, 400 tokens"},
            "synthetic_files": [os.path.basename(f) for f in synth_files],
            "versions": package_versions("d2vi", "transformers", "torch")}
json.dump(manifest, open(out.replace(".parquet", ".manifest.json"), "w"), indent=2)
from google.colab import files
files.download(out); files.download(out.replace(".parquet", ".manifest.json"))
